# Debugging Techniques

When AI code doesn't work, here's how to find and fix the problem.

## 1. Print Debugging

In [ ]:
# Simple but effective
def process_data(data):
    print(f"DEBUG: Input data = {data}")  # What did we receive?
    print(f"DEBUG: Type = {type(data)}")  # What type is it?
    
    result = []
    for item in data:
        print(f"DEBUG: Processing item = {item}")
        processed = item * 2
        print(f"DEBUG: Result = {processed}")
        result.append(processed)
    
    print(f"DEBUG: Final result = {result}")
    return result

process_data([1, 2, 3])

In [ ]:
# f-string debugging (Python 3.8+)
x = 42
y = "hello"
z = [1, 2, 3]

# The = shows variable name AND value
print(f"{x=}")
print(f"{y=}")
print(f"{z=}")
print(f"{len(z)=}")

## 2. Using `breakpoint()` (Python 3.7+)

In [ ]:
# In a script, breakpoint() opens interactive debugger
def calculate(a, b):
    result = a + b
    # breakpoint()  # Uncomment to debug - opens pdb
    # In debugger, you can:
    #   n - next line
    #   s - step into function
    #   c - continue execution
    #   p variable - print variable
    #   q - quit debugger
    return result * 2

print(calculate(3, 4))

## 3. Assert Statements

In [ ]:
# Assert checks assumptions
def divide(a, b):
    assert b != 0, "Divisor cannot be zero"
    assert isinstance(a, (int, float)), f"Expected number, got {type(a)}"
    return a / b

print(divide(10, 2))  # Works

try:
    divide(10, 0)
except AssertionError as e:
    print(f"Assertion failed: {e}")

In [ ]:
# Use asserts to document assumptions
def process_user(user):
    assert user is not None, "User cannot be None"
    assert 'name' in user, "User must have 'name' field"
    assert isinstance(user.get('age'), int), "Age must be integer"
    
    return f"Processing {user['name']}, age {user['age']}"

# These fail fast with clear messages
try:
    process_user({'name': 'Alice'})  # Missing age
except AssertionError as e:
    print(f"Failed: {e}")

## 4. Logging (Better than Print)

In [ ]:
import logging

# Configure logging
logging.basicConfig(
    level=logging.DEBUG,
    format='%(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def process_data(data):
    logger.debug(f"Input: {data}")
    logger.info(f"Processing {len(data)} items")
    
    try:
        result = [x * 2 for x in data]
        logger.debug(f"Result: {result}")
        return result
    except Exception as e:
        logger.error(f"Failed to process: {e}")
        raise

process_data([1, 2, 3])

In [ ]:
# Logging levels
logger.debug("Detailed info for debugging")
logger.info("General info about progress")
logger.warning("Something unexpected but not an error")
logger.error("Something failed")
logger.critical("Serious error, program may crash")

## 5. Inspect Objects

In [ ]:
# Understand what you're working with
mystery_object = {"name": "Alice", "scores": [90, 85]}

print(f"Type: {type(mystery_object)}")
print(f"Dir: {[x for x in dir(mystery_object) if not x.startswith('_')][:10]}")
print(f"Has 'get': {hasattr(mystery_object, 'get')}")
print(f"Is dict: {isinstance(mystery_object, dict)}")

In [ ]:
# For classes, inspect attributes
class User:
    def __init__(self, name):
        self.name = name
        self._private = "secret"

user = User("Alice")

# See all attributes
print(f"__dict__: {user.__dict__}")

# Check specific attribute
print(f"Has 'name': {hasattr(user, 'name')}")
print(f"Get 'name': {getattr(user, 'name', 'default')}")

## 6. Isolate the Problem

In [ ]:
# Binary search debugging: comment out half the code
def complex_function(data):
    # Step 1
    result = [x for x in data if x > 0]
    print(f"After step 1: {result}")
    
    # Step 2 - comment this out to isolate
    result = [x * 2 for x in result]
    print(f"After step 2: {result}")
    
    # Step 3 - comment this out to isolate
    result = sorted(result, reverse=True)
    print(f"After step 3: {result}")
    
    return result

complex_function([-1, 3, 0, 2, 1])

In [ ]:
# Create minimal reproduction
# Instead of debugging with real data, simplify:

# Real data might be huge and complex
# real_data = load_from_database()  

# Create minimal example that shows the bug
minimal_data = [{"id": 1, "value": None}]  # Simplified!

def process(data):
    return [item["value"] * 2 for item in data]  # Bug with None!

try:
    process(minimal_data)
except TypeError as e:
    print(f"Found the bug: {e}")

## 7. Check Types with Type Hints

In [ ]:
# Type hints + mypy catch bugs before runtime
from typing import List, Optional, Dict

def process_users(users: List[Dict[str, str]]) -> List[str]:
    """Process user list and return names."""
    return [user['name'] for user in users]

# Run mypy to check: mypy script.py
# It would catch: process_users("wrong type")

result = process_users([{"name": "Alice"}, {"name": "Bob"}])
print(result)

## 8. Common Debugging Patterns

In [ ]:
# Pattern 1: Debug decorator
from functools import wraps

def debug(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        print(f"  args: {args}")
        print(f"  kwargs: {kwargs}")
        result = func(*args, **kwargs)
        print(f"  returned: {result}")
        return result
    return wrapper

@debug
def add(a, b):
    return a + b

add(2, 3)

In [ ]:
# Pattern 2: Context manager for timing
import time
from contextlib import contextmanager

@contextmanager
def timer(name="Block"):
    start = time.time()
    yield
    elapsed = time.time() - start
    print(f"{name} took {elapsed:.4f}s")

with timer("Processing"):
    total = sum(range(1000000))

In [ ]:
# Pattern 3: Safe execution wrapper
def safe_execute(func, *args, default=None, **kwargs):
    """Execute function, return default on error."""
    try:
        return func(*args, **kwargs)
    except Exception as e:
        print(f"Error in {func.__name__}: {e}")
        return default

def risky_function(x):
    return 10 / x

result = safe_execute(risky_function, 0, default=0)
print(f"Result: {result}")

## 9. Debugging with AI Assistance

In [ ]:
# When asking AI to help debug, provide:
# 1. The error message
# 2. The relevant code
# 3. What you expected
# 4. What actually happened

# Example prompt:
"""
I'm getting this error:
```
TypeError: can only concatenate str (not "int") to str
```

Here's my code:
```python
name = "Alice"
age = 30
message = "Hello, " + name + ". You are " + age + " years old."
```

Expected: A greeting message
Actual: TypeError

How do I fix this?
"""

# Good AI answer would explain str() or f-strings
print("Provide context for better AI help!")

## Summary: Debugging Checklist

1. **Read the error message** - Often tells you exactly what's wrong
2. **Add print statements** - See what values actually are
3. **Check types** - `type(x)` and `isinstance(x, Type)`
4. **Simplify** - Create minimal reproduction
5. **Binary search** - Comment out code to isolate
6. **Check assumptions** - Add asserts
7. **Use logging** - Better than print for production
8. **Ask AI** - With full context and error messages

## Next Up

Reading and understanding tracebacks.

Continue to: [Reading Tracebacks](03-reading-tracebacks.ipynb)